In [2]:
!pip install "numpy<2"
!pip install surprise -qqq

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [21]:
import dask.dataframe as dd; import pandas as pd; import numpy as np;
from surprise import SVD, Dataset, Reader, accuracy;
from surprise.model_selection import train_test_split;
from sklearn.feature_extraction.text import TfidfVectorizer;
from sklearn.metrics.pairwise import linear_kernel;
from sentence_transformers import SentenceTransformer;
from sklearn.metrics import mean_squared_error, r2_score, median_absolute_error;
import torch;

In [4]:
import gdown, zipfile
url = 'https://drive.google.com/file/d/1xEOrywb5y_P8JedxeN0ICsASs2zzDgdV/view?usp=sharing'
path = gdown.download(url, quiet=False, fuzzy=True)
with zipfile.ZipFile(path, 'r') as f: f.extractall('data')

Downloading...
From (original): https://drive.google.com/uc?id=1xEOrywb5y_P8JedxeN0ICsASs2zzDgdV
From (redirected): https://drive.google.com/uc?id=1xEOrywb5y_P8JedxeN0ICsASs2zzDgdV&confirm=t&uuid=7ebbb591-bce2-4575-865c-46a09b15c7b1
To: /content/ml-latest.zip
100%|██████████| 351M/351M [00:02<00:00, 118MB/s]


#Часть 1

##1 + 2. Подготовка данных / Предобработка данных


In [5]:
r_d = dd.read_csv('/content/data/ml-latest/ratings.csv').dropna();
m_d = dd.read_csv('/content/data/ml-latest/movies.csv').dropna();

R = r_d.sample(frac=0.01).compute(); M = m_d.compute();

# Центрирование: вычитаем среднее пользователя для устранения разницы в шкалах
R['r_c'] = R['rating'] - R.groupby('userId')['rating'].transform('mean');

# Подготовка данных для SVD (на центрированных оценках)
rd = Reader(rating_scale=(-5.0, 5.0));
dt = Dataset.load_from_df(R[['userId', 'movieId', 'r_c']], rd);
tr, te = train_test_split(dt, test_size=0.2)

In [6]:
disp=R
display(disp.head(5));display(disp.describe());print(disp.info())

,userId,movieId,rating,timestamp,r_c
203891,2102,3770,2.0,965838293,-0.583333
1740524,17093,2023,3.0,1135475153,0.000000
1825347,17929,4973,4.0,1504555037,0.000000
945469,9068,180539,3.5,1578753316,0.000000
986918,9448,2455,3.5,1428992686,0.571429


,userId,movieId,rating,timestamp,r_c
count,338321.000000,338321.000000,338321.000000,3.383210e+05,3.383210e+05
mean,165437.689334,28273.707033,3.537905,1.269816e+09,7.455720e-19
std,95316.992973,49898.979613,1.063890,2.539853e+08,7.414097e-01
min,2.000000,1.000000,0.500000,8.231852e+08,-3.984848e+00
25%,83080.000000,1220.000000,3.000000,1.047230e+09,-3.333333e-01
50%,166104.000000,3264.000000,3.500000,1.265874e+09,0.000000e+00
75%,247533.000000,39446.000000,4.000000,1.496915e+09,4.000000e-01
max,330975.000000,288443.000000,5.000000,1.689832e+09,3.825000e+00


<class 'pandas.core.frame.DataFrame'>
Index: 338321 entries, 203891 to 2072955
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     338321 non-null  int64  
 1   movieId    338321 non-null  int64  
 2   rating     338321 non-null  float64
 3   timestamp  338321 non-null  int64  
 4   r_c        338321 non-null  float64
dtypes: float64(2), int64(3)
memory usage: 15.5 MB
None


In [15]:
disp=M
display(disp.head(5));display(disp.describe());print(disp.info())

,movieId,title,genres,g
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,Comedy


,movieId
count,86537.000000
mean,155932.817096
std,78037.145347
min,1.000000
25%,111443.000000
50%,164574.000000
75%,211225.000000
max,288983.000000


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86537 entries, 0 to 86536
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  86537 non-null  int64 
 1   title    86537 non-null  string
 2   genres   86537 non-null  string
 3   g        86537 non-null  string
dtypes: int64(1), string(3)
memory usage: 6.9 MB
None


##3. Реализация SVD

In [7]:
a_c = SVD(); a_c.fit(tr);
p_c = a_c.test(te);
rmse_svd = accuracy.rmse(p_c, verbose=False);
mae_svd = accuracy.mae(p_c, verbose=False);

##4. Оценка рекомендательной системы

In [8]:
print(f'RMSE: {rmse_svd}')
print(f'MAE: {mae_svd}')

RMSE: 0.7538208909458592
MAE: 0.5437770447067652


##5. Анализ результатов

In [9]:
df_an = pd.DataFrame([(p.uid, p.iid, p.r_ui, p.est) for p in p_c], columns=['User', 'Movie', 'Real', 'Pred']);
df_an['Error'] = abs(df_an.Real - df_an.Pred);
print("Наиболее точные:"); display(df_an.sort_values('Error').head(5));
print("Наименее точные:"); display(df_an.sort_values('Error', ascending=False).head(5));

Наиболее точные:


,User,Movie,Real,Pred,Error
18713,179385,33145,0.000000,-0.000009,0.000009
63139,265021,134170,0.000000,-0.000032,0.000032
63338,21725,177765,0.166667,0.166634,0.000033
22043,12798,3186,-0.250000,-0.249931,0.000069
63648,291208,51709,0.200000,0.200073,0.000073


Наименее точные:


,User,Movie,Real,Pred,Error
19339,190452,107406,3.727273,-0.358098,4.085371
31126,81438,2701,3.000000,-1.011004,4.011004
59223,190452,79545,3.727273,-0.228661,3.955934
62163,284655,122920,3.755556,-0.169406,3.924962
60092,284655,33649,3.755556,-0.047796,3.803352


#Часть 2

##1. Подготовка данных

In [10]:
# Данные R и M уже загружены в 1 части.
tr, te = train_test_split(dt, test_size=0.2);

##2. Реализация коллаборативной фильтрации с использованием SVD

In [19]:
algo = SVD(); algo.fit(tr); pr_s = algo.test(te);

##3. Реализация контентной фильтрации

In [16]:
M['g'] = M['genres'].str.replace('|', ' ');
# 3a. TF-IDF
v_t = TfidfVectorizer(stop_words='english'); mx_t = v_t.fit_transform(M['g']);
# 3b. BERT (GPU Optimized)
dev = 'cuda' if torch.cuda.is_available() else 'cpu';
st = SentenceTransformer('all-MiniLM-L6-v2', device=dev);
u_g = M['g'].unique(); u_e = st.encode(u_g, batch_size=256, show_progress_bar=False);
g_m = dict(zip(u_g, u_e)); mx_e = np.stack(M['g'].map(g_m).values);
i_d = pd.Series(M.index, index=M['movieId']).drop_duplicates();

def get_sc(u, i, R, mx, i_d, is_tf=False):
    if i not in i_d: return 0.0
    ix_i = i_d[i]; ur = R[R['userId'] == u]
    if ur.empty: return 0.0
    sc, wt = 0, 0
    for _, r in ur.iterrows():
        if r['movieId'] in i_d:
            ix_j = i_d[r['movieId']];
            w = (mx[ix_i] @ mx[ix_j].T).toarray()[0][0] if is_tf else np.dot(mx[ix_i], mx[ix_j]);
            sc += w * r['r_c']; wt += w
    return sc / wt if wt > 0 else 0.0

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 235 stored elements and shape (100, 23)>

##4. Объединение моделей


In [17]:
def hb(u, i, mx, is_tf=False, w=0.7):
    return w * algo.predict(u, i).est + (1 - w) * get_sc(u, i, R, mx, i_d, is_tf)

##5. Оценка гибридной модели

In [22]:
t_s = pr_s[:100]; r_t = np.array([o.r_ui for o in t_s]);
p_lists = {
    'SVD (Только коллаборация)': [o.est for o in t_s],
    'SVD + TF-IDF (Гибрид)': [hb(o.uid, o.iid, mx_t, is_tf=True) for o in t_s],
    'SVD + BERT Embeddings (Гибрид)': [hb(o.uid, o.iid, mx_e, is_tf=False) for o in t_s]
};

def calc_metrics(p):
    p = np.array(p);
    return [
        np.sqrt(mean_squared_error(r_t, p)), # RMSE
        mean_squared_error(r_t, p),          # MSE
        np.mean(np.abs(r_t - p)),            # MAE
        median_absolute_error(r_t, p),       # MedAE
        r2_score(r_t, p)                     # R2
    ];

# Сводная таблица
res_data = {name: calc_metrics(preds) for name, preds in p_lists.items()};
cols = [
    'RMSE (Кв. корень из ошибки)', 'MSE (Среднеквадратичная)',
    'MAE (Средняя абсолютная)', 'MedAE (Медианная абс.)', 'R² (Коэф. детерминации)'
];
res_df = pd.DataFrame.from_dict(res_data, orient='index', columns=cols);

print("\nСРАВНИТЕЛЬНЫЙ АНАЛИЗ ВСЕХ ПОДХОДОВ:");
display(res_df.style.highlight_min(subset=cols[:-1], color='lightgreen', axis=0)\
                  .highlight_max(subset=[cols[-1]], color='lightgreen', axis=0));


СРАВНИТЕЛЬНЫЙ АНАЛИЗ ВСЕХ ПОДХОДОВ:


,RMSE (Кв. корень из ошибки),MSE (Среднеквадратичная),MAE (Средняя абсолютная),MedAE (Медианная абс.),R² (Коэф. детерминации)
SVD (Только коллаборация),0.799155,0.638648,0.567793,0.390758,-0.035146
SVD + TF-IDF (Гибрид),0.690588,0.476912,0.479970,0.312545,0.227003
SVD + BERT Embeddings (Гибрид),0.753426,0.567651,0.523187,0.349251,0.079930
